# BrainDecoding — Google Colab CPU 缓存预热与单文件归档工作流

本 Notebook 运行在 **Google Colab CPU 实例** 上，负责将 Google Drive 上的原始脑电/脑磁数据（`ChineseEEG-2` 或 `Pallier2025`）预处理到 Colab 本地高速盘 `/content/local_cache`，随后打包为云盘上的单一 `.tar` 文件（`archives/features_<modality>.tar`），避免在昂贵的 GPU 实例上现场重算或遭遇 Google Drive FUSE 零散小文件 I/O 瓶颈。

### 存储与路径分工
- **代码仓库**：`/content/code`
- **虚拟环境**：`/content/proj-venv`（保留云端原生 PyTorch/CUDA 版本约束）
- **原始数据软链接桥**：`/content/studies_links` -> `/content/drive/MyDrive/BrainDecoding/data`（规避 Drive FUSE `Errno 95` 并锁定缓存哈希）
- **本地高速缓存**：`/content/local_cache`
- **云盘归档输出**：`/content/drive/MyDrive/BrainDecoding/archives/features_<modality>.tar`

In [ ]:
# 1. 挂载 Google Drive 并配置核心路径与参数
import os
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('非 Colab 环境，跳过 drive.mount')

# ==================== 用户可修改参数 ====================
GIT_REPO_URL = 'https://github.com/YOUR_USERNAME/BrainDecoding.git'  # 替换为你的 Git 仓库地址
GIT_BRANCH = 'main'
DRIVE_BASE_DIR = '/content/drive/MyDrive/BrainDecoding'
DATASET = 'chineseeeg2_littleprince'  # 可选: 'chineseeeg2_littleprince' 或 'pallier2025'
MODALITY = 'eeg'                      # 归档标识: 'eeg' 或 'meg'
SUBSET_ARGS = '--subject all'         # 或 '--num-subjects 1' / '--subjects sub-01,sub-02'
# ========================================================

CODE_DIR = '/content/code'
VENV_DIR = '/content/proj-venv'
PYTHON = f'{VENV_DIR}/bin/python'
LINKS_DIR = '/content/studies_links'
LOCAL_CACHE_DIR = '/content/local_cache'

DRIVE_DATA_DIR = f'{DRIVE_BASE_DIR}/data'
DRIVE_ARCHIVE_DIR = f'{DRIVE_BASE_DIR}/archives'
DRIVE_RESULTS_DIR = f'{DRIVE_BASE_DIR}/runs'

for folder in (LINKS_DIR, LOCAL_CACHE_DIR, DRIVE_ARCHIVE_DIR, DRIVE_RESULTS_DIR):
    Path(folder).mkdir(parents=True, exist_ok=True)

# 统一通过环境变量驱动所有路径（保证 CPU 预热与 GPU 训练哈希 100% 一致）
os.environ['BRAINDECODING_DATA_DIR'] = LINKS_DIR
os.environ['BRAINDATA_ROOT'] = LINKS_DIR
os.environ['BRAINDECODING_CACHE_DIR'] = LOCAL_CACHE_DIR
os.environ['BRAINDECODING_RESULTS_DIR'] = DRIVE_RESULTS_DIR
os.environ['BRAINDECODING_MODEL_DIR'] = f'{DRIVE_BASE_DIR}/models'

print('BRAINDECODING_DATA_DIR    =', os.environ['BRAINDECODING_DATA_DIR'])
print('BRAINDECODING_CACHE_DIR   =', os.environ['BRAINDECODING_CACHE_DIR'])
print('BRAINDECODING_RESULTS_DIR =', os.environ['BRAINDECODING_RESULTS_DIR'])

In [ ]:
# 2. 克隆或更新最新代码到 /content/code
!if [ ! -d "{CODE_DIR}/.git" ]; then \
    git clone -b {GIT_BRANCH} {GIT_REPO_URL} {CODE_DIR}; \
else \
    git -C {CODE_DIR} fetch origin && git -C {CODE_DIR} checkout {GIT_BRANCH} && git -C {CODE_DIR} pull --ff-only; \
fi

In [ ]:
# 3. 防崩版初始化虚拟环境 + 建立本地 /content/studies_links 软链接桥（流式无缓冲输出）
!python3 -u {CODE_DIR}/scripts/colab_setup.py \
    --venv-dir {VENV_DIR} \
    --project-root {CODE_DIR} \
    --drive-data-dir {DRIVE_DATA_DIR} \
    --links-dir {LINKS_DIR} \
    --local-cache-dir {LOCAL_CACHE_DIR}

In [ ]:
# 4. 在本地高速盘 /content/local_cache 预热特征缓存（流式无缓冲输出）
!cd {CODE_DIR} && {PYTHON} -u -m braindecoding cache \
    --dataset {DATASET} \
    {SUBSET_ARGS}

In [ ]:
# 5. 将 /content/local_cache 中的零散缓存碎片打包为 Google Drive 上的单一 .tar 归档
!cd {CODE_DIR} && {PYTHON} -u scripts/cache_archive.py pack \
    --cache-dir {LOCAL_CACHE_DIR} \
    --archive-dir {DRIVE_ARCHIVE_DIR} \
    --modality {MODALITY}